# 01 — Data Exploration

First step before touching any model — just look at the data.
Figure out what's in each table, what's missing, and what patterns stand out.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# walks up from cwd until it finds the folder with src/ and requirements.txt
def find_project_root(start: Path, depth: int = 5) -> Path:
    path = start.resolve()
    for _ in range(depth):
        if (path / "src").exists() and (path / "requirements.txt").exists():
            return path
        path = path.parent
    raise RuntimeError(f"Can't find project root from {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_raw_tables, build_master_df, add_parsed_lap_times

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)
print("Root:", PROJECT_ROOT)

## Raw tables overview

In [ ]:
tables = load_raw_tables()

for name, df in tables.items():
    print(f"{name:30s}  {df.shape}")

## results.csv — the target table

One row per driver per race. `positionOrder` is what we'll predict.

In [ ]:
results = tables["results"]
print(results.dtypes)
print()
results.head(10)

In [ ]:
# check what's actually missing
missing = results.isnull().sum()
print(missing[missing > 0])

## Grid position vs podium

Starting from pole is a massive advantage — let's see how big.

In [ ]:
df = build_master_df(tables)
df = add_parsed_lap_times(df)

df["is_podium"]     = (df["positionOrder"] <= 3).astype(int)
df["grid"]          = pd.to_numeric(df["grid"], errors="coerce")
df["grid_position"] = df["grid"].replace(0, 20)  # pit lane start -> treat as last

podium_by_grid = (
    df[df["grid_position"].between(1, 20)]
    .groupby("grid_position")["is_podium"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(podium_by_grid["grid_position"], podium_by_grid["is_podium"], color="steelblue")
ax.set_xlabel("Grid Position")
ax.set_ylabel("Podium Rate")
ax.set_title("Podium Rate by Grid Position (1950-2024)")
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "podium_rate_by_grid.png", dpi=150)
plt.show()

## Class balance

Only ~13% of entries are podiums (3 out of ~20 cars). This imbalance means we can't just use raw accuracy as a metric.

In [ ]:
balance = df["is_podium"].value_counts(normalize=True)
print(balance.rename({0: "No Podium", 1: "Podium"}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No Podium", "Podium"], balance.values, color=["#e74c3c", "#2ecc71"])
ax.set_ylabel("Proportion")
ax.set_title("Class Balance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png", dpi=150)
plt.show()

## Qualifying data availability

Q1/Q2/Q3 format started in 2006. Anything before that has no qualifying times.

In [ ]:
for col in ["q1_seconds", "q2_seconds", "q3_seconds"]:
    if col in df.columns:
        pct = df[col].isna().mean()
        print(f"{col}: {pct:.1%} missing")

## Constructor dominance

The car matters as much as the driver in F1.

In [ ]:
constructor_wins = (
    df[df["positionOrder"] == 1]
    .groupby("constructorRef")
    .size()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
constructor_wins.plot(kind="bar", ax=ax, color="tomato")
ax.set_xlabel("Constructor")
ax.set_ylabel("Wins")
ax.set_title("Top 15 Constructors by Wins (1950-2024)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constructor_wins.png", dpi=150)
plt.show()

## Races per season

Grew from 7 in 1950 to 24 recently — relevant for rolling features.

In [ ]:
races_per_year = df.groupby("year")["raceId"].nunique()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(races_per_year.index, races_per_year.values,
        marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Year")
ax.set_ylabel("Races")
ax.set_title("F1 Races Per Season")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "races_per_year.png", dpi=150)
plt.show()

print("done — figures at:", FIGURES_DIR)